# PHẦN 2: TRỰC QUAN HÓA VÀ PHÂN TÍCH DỮ LIỆU
## YÊU CẦU 2: EDA

### 1. Chiến lược Kết hợp Dữ liệu (Data Merging)
Sử dụng bảng `orders` làm gốc và thực hiện các phép nối (`Left Join`) để mở rộng thông tin:
* **Chi tiết đơn hàng & Sản phẩm:** Kết nối `order_items` và `products` để lấy thông tin về giá bán, giá vốn (COGS) và phân loại hàng hóa.
* **Thông tin khách hàng:** Kết nối với bảng `customers` để phân tích hành vi theo giới tính, độ tuổi và khu vực.
* **Vận chuyển & Trả hàng:** Kết nối với `shipments` và `returns` để theo dõi hiệu quả hậu cần và chất lượng sản phẩm.
* **Thanh toán:** Kết nối với bảng `payments` để xác nhận giá trị giao dịch thực tế.

### 2. Xây dựng các Chỉ số Đo lường (Feature Engineering)
Để phục vụ phân tích đa chiều, các biến số mới được tính toán bao gồm:
* **Doanh thu (Revenue):** Tính toán dựa trên đơn giá và số lượng sau khi đã trừ các khoản chiết khấu.
* **Lợi nhuận gộp (Gross Profit):** Hiệu số giữa doanh thu và tổng giá vốn hàng bán.
* **Thời gian giao hàng (Delivery Time):** Số ngày chênh lệch từ lúc đặt hàng đến khi khách hàng nhận được máy.
* **Tỷ lệ trả hàng (Return Rate):** Biến phân loại (0/1) để xác định các đơn hàng bị hoàn trả.
* **Giá trị vòng đời khách hàng (CLV):** Tổng giá trị đóng góp của mỗi khách hàng dựa trên lịch sử giao dịch.

### 3. Mục tiêu đầu ra
* Tạo ra file `master_eda_dataset.csv` chứa toàn bộ thông tin cần thiết.
* Đảm bảo tính nhất quán dữ liệu cho nhóm Trực quan hóa (Visualization) và Xây dựng mô hình (Modeling).

In [1]:
import pandas as pd
import numpy as np
import os

# 1. Cấu hình đường dẫn
CLEANED_DIR = 'cleaned_data/'
OUTPUT_FILE = 'cleaned_data/master_eda_dataset.csv'

def load_df(name):
    return pd.read_csv(os.path.join(CLEANED_DIR, f'cleaned_{name}.csv'))

# 2. Tải các bảng quan trọng
orders = load_df('orders')
order_items = load_df('order_items')
products = load_df('products')
customers = load_df('customers')
payments = load_df('payments')
returns = load_df('returns')
shipments = load_df('shipments')

def create_master_dataset():
    print("🚀 Đang bắt đầu quá trình Merge dữ liệu...")
    
    # Merge Orders với Order Items
    master = pd.merge(orders, order_items, on='order_id', how='left')
    
    # Merge với Products để lấy COGS và Category
    master = pd.merge(master, products, on='product_id', how='left')
    
    # Merge với Customers để lấy thông tin người mua
    master = pd.merge(master, customers, on='customer_id', how='left')
    
    # Merge với Payments
    master = pd.merge(master, payments, on='order_id', how='left')
    
    # Merge với Shipments
    master = pd.merge(master, shipments, on='order_id', how='left')
    
    # Merge với Returns (Đánh dấu đơn bị trả)
    master = pd.merge(master, returns, on='order_id', how='left', indicator='is_returned')
    master['is_returned'] = master['is_returned'].apply(lambda x: 1 if x == 'both' else 0)

    print(f"✅ Merge hoàn tất. Kích thước Master Dataset: {master.shape}")

    # 3. TẠO CÁC FEATURE HỖ TRỢ PHÂN TÍCH
    print("📊 Đang tính toán các chỉ số kinh doanh...")
    
    # Chuyển đổi lại ngày tháng (đề phòng khi load csv bị mất định dạng)
    date_cols = ['order_date', 'delivery_date', 'ship_date']
    for col in date_cols:
        if col in master.columns:
            master[col] = pd.to_datetime(master[col])

    # A. Doanh thu & Lợi nhuận
    # Revenue = (Giá bán - Giảm giá) * Số lượng
    master['revenue'] = (master['price'] - master.get('discount_value', 0)) * master['quantity']
    # Gross Profit = Doanh thu - (Giá vốn * Số lượng)
    master['gross_profit'] = master['revenue'] - (master['cogs'] * master['quantity'])
    
    # B. Thời gian giao hàng (Delivery Time tính theo ngày)
    master['delivery_time'] = (master['delivery_date'] - master['order_date']).dt.days
    
    # C. Monthly Sales (Tạo cột Tháng-Năm để vẽ biểu đồ xu hướng)
    master['month_year'] = master['order_date'].dt.to_period('M')

    # D. Customer Lifetime Value (CLV) sơ bộ - Tổng chi tiêu của mỗi khách
    clv = master.groupby('customer_id')['revenue'].transform('sum')
    master['customer_total_spend'] = clv

    # 4. LƯU DỮ LIỆU TRUNG GIAN
    master.to_csv(OUTPUT_FILE, index=False)
    print(f"💾 Đã lưu Master Dataset tại: {OUTPUT_FILE}")
    
    return master

if __name__ == "__main__":
    df_master = create_master_dataset()
    
    # Hiển thị vài dòng đầu để kiểm tra
    print("\n--- Xem trước dữ liệu EDA ---")
    print(df_master[['order_id', 'revenue', 'gross_profit', 'is_returned', 'delivery_time']].head())

🚀 Đang bắt đầu quá trình Merge dữ liệu...
✅ Merge hoàn tất. Kích thước Master Dataset: (722575, 40)
📊 Đang tính toán các chỉ số kinh doanh...
💾 Đã lưu Master Dataset tại: cleaned_data/master_eda_dataset.csv

--- Xem trước dữ liệu EDA ---
   order_id       revenue  gross_profit  is_returned  delivery_time
0         1   7764.827430    388.241372            0            7.0
1         2  72985.997235  10072.067618            1            6.0
2         3  33085.286084   2812.249317            0            3.0
3         4  53726.102941   7698.950551            0            7.0
4         6   1609.911509    561.215152            0           10.0
